# Lab 2: Kafka + Spark Structured Streaming

## Cel

Przetwarzamy ten sam strumień transakcji z Lab 1, ale teraz w Apache Spark:

- Połączenie Sparka z Kafką,
- Okna tumbling i sliding,
- Transformacje bezstanowe i stanowe na skalę.

**Case biznesowy (ciąg dalszy):** System monitoringu musi generować raporty okienkowe — np. łączny przychód per sklep co 5 minut.

---

## Część 1: Podstawy Sparka (tryb batch)

### Zadanie 1.1 — Utwórz SparkSession

In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
    .appName("Lab2")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark: {spark.version}")

### Zadanie 1.2 — Utwórz DataFrame z przykładowych danych

Utwórz Spark DataFrame o tej samej strukturze co transakcje z Kafki. Użyj `StructType`.

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.sql.functions import to_timestamp

data = [
    ("TX001", "u05", 150.0, "Warszawa", "elektronika", "2026-04-01 10:00:00"),
    ("TX002", "u12", 4500.0, "Kraków", "odzież", "2026-04-01 10:01:00"),
    ("TX003", "u05", 89.0, "Warszawa", "żywność", "2026-04-01 10:02:30"),
    ("TX004", "u08", 7800.0, "Gdańsk", "elektronika", "2026-04-01 10:03:00"),
    ("TX005", "u12", 55.0, "Kraków", "książki", "2026-04-01 10:05:00"),
    ("TX006", "u03", 1200.0, "Wrocław", "elektronika", "2026-04-01 10:06:30"),
]

# TWÓJ KOD
# 1. Zdefiniuj schemat
# 2. Utwórz DataFrame
# 3. Skonwertuj kolumnę timestamp
# 4. Pokaż df.show() i df.printSchema()

### Zadanie 1.3 — Tumbling window na danych batch

Pogrupuj transakcje w okna 3-minutowe. Policz transakcje i zsumuj kwoty per okno.

In [ ]:
import pyspark.sql.functions as F

# TWÓJ KOD
# Użyj F.window("kolumna_czasu", "3 minutes")

---

## Część 2: Streaming ze źródła `rate`

### Zadanie 2.1 — Pierwszy strumień

In [ ]:
batch_counter = {"count": 0}

def process_batch(df, batch_id, max_batches=5):
    batch_counter["count"] += 1
    print(f"--- Batch {batch_id} ---")
    df.show(truncate=False)
    if batch_counter["count"] >= max_batches:
        raise Exception("Stop")

rate_df = (spark.readStream.format("rate").option("rowsPerSecond", 2).load())
rate_df.printSchema()

query = (rate_df.writeStream.format("console").outputMode("append")
    .foreachBatch(process_batch).start())
try:
    query.awaitTermination()
except:
    query.stop()

### Zadanie 2.2 — Wzbogać i filtruj strumień

Dodaj kolumny: `user_id`, `amount`, `store`. Filtruj `amount > 2000` (tryb append).

In [ ]:
from pyspark.sql.functions import expr
batch_counter["count"] = 0

# TWÓJ KOD

### Zadanie 2.3 — Tumbling window + watermark na strumieniu

Okna 10-sekundowe, watermark 5 sekund. Zlicz i zsumuj per okno per store.

In [ ]:
batch_counter["count"] = 0
# TWÓJ KOD

---

## Część 3: Odczyt z Kafki

### Zadanie 3.1 — Połącz Sparka z Kafką

Uruchom producenta z Lab 1 w terminalu.

In [ ]:
kafka_df = (spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("subscribe", "transactions")
    .option("startingOffsets", "earliest")
    .load())
kafka_df.printSchema()

### Zadanie 3.2 — Parsuj JSON z Kafki

In [ ]:
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

tx_schema = StructType([
    StructField("tx_id", StringType()),
    StructField("user_id", StringType()),
    StructField("amount", DoubleType()),
    StructField("store", StringType()),
    StructField("category", StringType()),
    StructField("timestamp", StringType()),
])

# TWÓJ KOD
# 1. col("value").cast("string")
# 2. from_json(..., tx_schema)
# 3. Wyciągnij pola
# 4. Skonwertuj timestamp

### Zadanie 3.3 — Agregacja okienkowa na strumieniu z Kafki

Okna 1-minutowe per sklep. Watermark 30 sekund. Policz transakcje, sumę, średnią.

In [ ]:
# TWÓJ KOD

---

## Praca domowa

1. Dodaj sliding window: 5 minut, krok 1 minuta, per kategoria.
2. Zapisz alerty (amount > 3000) do nowego tematu Kafka `alerts`.
3. Wypchnij kod do repozytorium Git.

**Następne zajęcia:** Reguły decyzyjne na danych strumieniowych.

In [ ]:
spark.stop()